# EMCRAG — valutazione RAGAS standalone M0 → M1

Notebook da eseguire in **VS Code / WSL**, kernel **conda_ok_14apr**.

Questo notebook non carica MedGemma e non usa la GPU. Legge le risposte M0/M1 già congelate, valuta soltanto **Response Relevancy** con RAGAS 0.2.5, GPT-4o e `text-embedding-3-small`, quindi esegue il confronto paired M1−M0.

Coorte primaria: i 194 casi eleggibili in entrambe le condizioni. Le chiamate completate vengono salvate progressivamente e non vengono ripetute.


## 1. Contratto e controllo degli input

La cella verifica il checksum del file paired, la numerosità della coorte e la presenza della chiave API senza mostrarla.


In [1]:
from pathlib import Path
from collections import Counter
import hashlib, importlib.metadata as mdlib, json, math, os, platform, sys, time

PROJECT_DIR = Path("/home/eugrug/Project_CRAG_MedGemma")
PAIRED_PATH = PROJECT_DIR / "benchmark_m0_m1" / "outputs" / "M0_M1_paired_196.jsonl"
RAGAS_OUTPUT_DIR = PROJECT_DIR / "benchmark_m0_m1" / "ragas_m0_m1_v07_03"
SCORES_PATH = RAGAS_OUTPUT_DIR / "response_relevancy_scores.jsonl"

EXPECTED_PAIRED_SHA256 = "b4a2dcbf38bdcad143b4cffcf4909de5ce59e588b8a4b505a4b058ec230d1bf6"
EXPECTED_CONTRACT_SHA256 = "7c16195744b17120e21bd6fb7fe57aa1c18eea3a4e99a2721d46a572df91dd85"
EXPECTED_JOINT_ELIGIBLE = 194
RAGAS_VERSION = "0.2.5"
JUDGE_MODEL = "gpt-4o"
EMBEDDING_MODEL = "text-embedding-3-small"
BOOTSTRAP_SEED = 20261005

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

require(PAIRED_PATH.is_file(), f"File paired assente: {PAIRED_PATH}")
require(sha256_file(PAIRED_PATH) == EXPECTED_PAIRED_SHA256, "Checksum paired diverso: STOP")
require(os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY non presente nell'ambiente: STOP")

paired_rows = [
    json.loads(line)
    for line in PAIRED_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

require(len(paired_rows) == 196, f"Record paired inattesi: {len(paired_rows)}")
require(len({r["sample_id"] for r in paired_rows}) == 196, "sample_id duplicati: STOP")
require(all(r.get("contract_sha256") == EXPECTED_CONTRACT_SHA256 for r in paired_rows), "Contratto diverso: STOP")

eligible_pairs = [r for r in paired_rows if r["M0_eligible"] and r["M1_eligible"]]
require(len(eligible_pairs) == EXPECTED_JOINT_ELIGIBLE, f"Pair eleggibili: {len(eligible_pairs)}")

RAGAS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("Paired SHA256:", EXPECTED_PAIRED_SHA256)
print("Pair totali:", len(paired_rows))
print("Pair congiuntamente eleggibili:", len(eligible_pairs))
print("Distribuzione:", Counter(r["dataset"] for r in eligible_pairs))
print("OPENAI_API_KEY: presente (valore non mostrato)")


Python: /home/eugrug/miniforge3/envs/conda_ok_14apr/bin/python
Paired SHA256: b4a2dcbf38bdcad143b4cffcf4909de5ce59e588b8a4b505a4b058ec230d1bf6
Pair totali: 196
Pair congiuntamente eleggibili: 194
Distribuzione: Counter({'IMB_Curated_Test_Set_1000': 99, 'MMMED_Italian': 95})
OPENAI_API_KEY: presente (valore non mostrato)


## 2. Configurazione RAGAS

La configurazione replica quella già impiegata per R0, limitandola alla metrica scientificamente appropriata per il confronto standalone: **Response Relevancy**.


In [8]:
import ragas
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import ResponseRelevancy

require(ragas.__version__ == RAGAS_VERSION, f"RAGAS {ragas.__version__}; richiesta {RAGAS_VERSION}: STOP")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
model_ids = {model.id for model in client.models.list().data}
require(JUDGE_MODEL in model_ids, f"{JUDGE_MODEL} non disponibile per questa API key: STOP")

# Was:
# judge = ChatOpenAI(model=JUDGE_MODEL, temperature=0, max_retries=2)
# This only raises the maximum threshold for the evaluator's output.
judge = ChatOpenAI(
    model=JUDGE_MODEL,
    temperature=0,
    max_tokens=4096,
    max_retries=2,
)

ragas_llm = LangchainLLMWrapper(judge)

print("Judge max_tokens:", judge.max_tokens)
      
embedding_client = OpenAIEmbeddings(model=EMBEDDING_MODEL)
ragas_embeddings = LangchainEmbeddingsWrapper(embedding_client)
response_relevancy_metric = ResponseRelevancy()

print("RAGAS:", ragas.__version__)
print("Judge:", JUDGE_MODEL)
print("Embeddings:", EMBEDDING_MODEL)
print("Metrica:", response_relevancy_metric.__class__.__name__)
print("Connessione API: PASS")


Judge max_tokens: 4096
RAGAS: 0.2.5
Judge: gpt-4o
Embeddings: text-embedding-3-small
Metrica: ResponseRelevancy
Connessione API: PASS


## 3. Runner riprendibile

Ogni blocco contiene pair completi M0/M1. I risultati vengono aggiunti al file JSONL soltanto dopo il completamento del blocco. Un eventuale riavvio salta tutte le valutazioni già salvate.


In [15]:
FAILURES_PATH = SCORES_PATH.parent / "response_relevancy_failures.jsonl"


def append_jsonl_durable(path, row):
    """Scrive un record JSONL e forza il salvataggio fisico su disco."""
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")
        f.flush()
        os.fsync(f.fileno())


def load_completed_scores():
    completed = {}

    if not SCORES_PATH.exists():
        return completed

    for line_number, line in enumerate(
        SCORES_PATH.read_text(encoding="utf-8").splitlines(), 1
    ):
        if not line.strip():
            continue

        row = json.loads(line)

        require(
            row.get("paired_sha256") == EXPECTED_PAIRED_SHA256,
            f"Checksum diverso alla riga {line_number}: STOP",
        )
        require(
            row.get("judge_model") == JUDGE_MODEL,
            f"Judge diverso alla riga {line_number}: STOP",
        )
        require(
            row.get("embedding_model") == EMBEDDING_MODEL,
            f"Embedding diverso alla riga {line_number}: STOP",
        )

        key = (row["sample_id"], row["condition"])
        require(key not in completed, f"Risultato duplicato: {key}")
        completed[key] = row

    return completed


def load_recorded_failures():
    failures = {}

    if not FAILURES_PATH.exists():
        return failures

    for line_number, line in enumerate(
        FAILURES_PATH.read_text(encoding="utf-8").splitlines(), 1
    ):
        if not line.strip():
            continue

        row = json.loads(line)

        require(
            row.get("paired_sha256") == EXPECTED_PAIRED_SHA256,
            f"Checksum failure diverso alla riga {line_number}: STOP",
        )
        require(
            row.get("judge_model") == JUDGE_MODEL,
            f"Judge failure diverso alla riga {line_number}: STOP",
        )
        require(
            row.get("embedding_model") == EMBEDDING_MODEL,
            f"Embedding failure diverso alla riga {line_number}: STOP",
        )

        key = (row["sample_id"], row["condition"])
        require(key not in failures, f"Failure duplicato: {key}")
        failures[key] = row

    return failures


def score_from_result(result_score):
    for key in ("answer_relevancy", "response_relevancy"):
        if key in result_score:
            value = float(result_score[key])
            require(
                math.isfinite(value),
                f"Score non finito: {result_score}",
            )
            return value, key

    raise RuntimeError(
        f"Colonna Response Relevancy non trovata: {result_score}"
    )


def pair_to_records(pair):
    common = {
        "user_input": pair["question"],
        "reference": pair["reference_answer"],
    }

    return [
        (
            {
                **common,
                "response": pair["M0_answer_for_evaluation"],
            },
            "M0",
        ),
        (
            {
                **common,
                "response": pair["M1_answer_for_evaluation"],
            },
            "M1",
        ),
    ]


def register_confirmed_imb_00791_failure(completed, failures):
    """
    Registra il fallimento M1 già verificato manualmente.

    Non viene assegnato uno score zero: il risultato RAGAS rimane missing.
    """
    key = ("imb_00791", "M1")

    if key in completed or key in failures:
        return

    matching_pairs = [
        pair
        for pair in eligible_pairs
        if pair["sample_id"] == "imb_00791"
    ]

    require(
        len(matching_pairs) == 1,
        "imb_00791 non trovato univocamente: STOP",
    )

    pair = matching_pairs[0]
    response = pair["M1_answer_for_evaluation"]

    # Controllo che sia effettivamente l'output degenerato osservato.
    require(
        len(response) == 509,
        "imb_00791/M1 non corrisponde all'output degenerato verificato: STOP",
    )
    require(
        response.startswith("2023-05-17-16-19-11-10-10-10"),
        "Pattern inatteso per imb_00791/M1: STOP",
    )

    failure_row = {
        "sample_id": pair["sample_id"],
        "benchmark_order": pair["benchmark_order"],
        "dataset": pair["dataset"],
        "execution_mode": pair["execution_mode"],
        "condition": "M1",
        "status": "evaluator_failure",
        "failure_type": "LLMDidNotFinishException",
        "failure_reason": (
            "Confirmed degenerate M1 generation; RAGAS question-generation "
            "judge reached its output-token limit at max_tokens=4096."
        ),
        "response_chars": len(response),
        "response_sha256": hashlib.sha256(
            response.encode("utf-8")
        ).hexdigest(),
        "score_imputed": False,
        "response_relevancy": None,
        "ragas_version": RAGAS_VERSION,
        "judge_model": JUDGE_MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "paired_sha256": EXPECTED_PAIRED_SHA256,
    }

    append_jsonl_durable(FAILURES_PATH, failure_row)
    failures[key] = failure_row

    print(
        "Failure confermato registrato: "
        "imb_00791/M1 — nessuno score imputato"
    )


def run_response_relevancy(max_new_pairs=None, pairs_per_chunk=1):
    """
    Valuta separatamente ogni condizione M0/M1.

    pairs_per_chunk è mantenuto per compatibilità con la cella 5,
    ma ciascuna valutazione viene salvata singolarmente.
    """
    require(pairs_per_chunk >= 1, "pairs_per_chunk deve essere >= 1")

    completed = load_completed_scores()
    failures = load_recorded_failures()

    register_confirmed_imb_00791_failure(completed, failures)

    pending_pairs = []

    for pair in eligible_pairs:
        sid = pair["sample_id"]

        needs_evaluation = any(
            (sid, condition) not in completed
            and (sid, condition) not in failures
            for condition in ("M0", "M1")
        )

        if needs_evaluation:
            pending_pairs.append(pair)

    if max_new_pairs is not None:
        pending_pairs = pending_pairs[:max_new_pairs]

    print(f"Score già salvati: {len(completed)}/388")
    print(f"Failure terminali registrati: {len(failures)}")
    print(f"Pair ancora da elaborare: {len(pending_pairs)}")

    for pair_number, pair in enumerate(pending_pairs, 1):
        sid = pair["sample_id"]

        for native_record, condition in pair_to_records(pair):
            key = (sid, condition)

            if key in completed or key in failures:
                continue

            metadata = {
                "sample_id": sid,
                "benchmark_order": pair["benchmark_order"],
                "dataset": pair["dataset"],
                "execution_mode": pair["execution_mode"],
                "condition": condition,
            }

            dataset = EvaluationDataset.from_list([native_record])
            started = time.perf_counter()

            try:
                result = evaluate(
                    dataset=dataset,
                    metrics=[response_relevancy_metric],
                    llm=ragas_llm,
                    embeddings=ragas_embeddings,
                    raise_exceptions=True,
                    in_ci=False,
                    show_progress=False,
                    batch_size=1,
                )
            except Exception as exc:
                raise RuntimeError(
                    f"Valutazione interrotta su {sid}/{condition}: "
                    f"{type(exc).__name__}: {exc}"
                ) from exc

            elapsed = round(time.perf_counter() - started, 3)

            require(
                len(result.scores) == 1,
                f"Numero score inatteso per {sid}/{condition}: STOP",
            )

            value, native_key = score_from_result(result.scores[0])

            output_row = {
                **metadata,
                "response_relevancy": value,
                "native_metric_key": native_key,
                "ragas_version": RAGAS_VERSION,
                "judge_model": JUDGE_MODEL,
                "embedding_model": EMBEDDING_MODEL,
                "paired_sha256": EXPECTED_PAIRED_SHA256,
            }

            append_jsonl_durable(SCORES_PATH, output_row)
            completed[key] = output_row

            print(
                f"[{pair_number}/{len(pending_pairs)}] "
                f"{sid}/{condition}: OK | "
                f"score={value:.6f} | "
                f"score salvati={len(completed)}/388 | "
                f"{elapsed:.1f}s",
                flush=True,
            )

    print("\nVALUTAZIONE TERMINATA")
    print(f"Score validi: {len(completed)}/388")
    print(f"Failure documentati: {len(failures)}")
    print(
        "Totale esiti contabilizzati:",
        len(completed) + len(failures),
        "/388",
    )

    return SCORES_PATH


print("Runner pronto.")
print("Score:", SCORES_PATH)
print("Failure:", FAILURES_PATH)

Runner pronto.
Score: /home/eugrug/Project_CRAG_MedGemma/benchmark_m0_m1/ragas_m0_m1_v07_03/response_relevancy_scores.jsonl
Failure: /home/eugrug/Project_CRAG_MedGemma/benchmark_m0_m1/ragas_m0_m1_v07_03/response_relevancy_failures.jsonl


## 4. Smoke test reale e riutilizzabile

Questa cella valuta un solo pair, quindi due risposte. Il risultato viene salvato e sarà riutilizzato dall’esecuzione completa.


In [4]:
SMOKE_OUTPUT_PATH = run_response_relevancy(max_new_pairs=1, pairs_per_chunk=1)
print("Smoke test completato:", SMOKE_OUTPUT_PATH)


Valutazioni già salvate: 0/388
Nuovi pair selezionati: 1


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Batch 1/2:   0%|          | 0/1 [00:00<?, ?it/s]

Pair completati nel blocco: 1 | totale score salvati: 2/388 | 16.0s
Smoke test completato: /home/eugrug/Project_CRAG_MedGemma/benchmark_m0_m1/ragas_m0_m1_v07_03/response_relevancy_scores.jsonl


## 5. Valutazione completa

Eseguire questa cella soltanto dopo il PASS dello smoke test. Riprende automaticamente dal pair successivo.


In [16]:
FULL_OUTPUT_PATH = run_response_relevancy(max_new_pairs=None, pairs_per_chunk=1)
print("Valutazione completa o ripresa terminata:", FULL_OUTPUT_PATH)


Failure confermato registrato: imb_00791/M1 — nessuno score imputato
Score già salvati: 162/388
Failure terminali registrati: 1
Pair ancora da elaborare: 113
[1/113] imb_00791/M0: OK | score=0.690368 | score salvati=163/388 | 9.5s
[2/113] imb_00296/M0: OK | score=0.743475 | score salvati=164/388 | 7.0s
[2/113] imb_00296/M1: OK | score=0.120961 | score salvati=165/388 | 8.6s
[3/113] imb_00081/M0: OK | score=0.469619 | score salvati=166/388 | 4.8s
[3/113] imb_00081/M1: OK | score=0.108934 | score salvati=167/388 | 7.2s
[4/113] imb_00875/M0: OK | score=0.469957 | score salvati=168/388 | 5.2s
[4/113] imb_00875/M1: OK | score=0.000000 | score salvati=169/388 | 6.2s
[5/113] imb_00978/M0: OK | score=0.562881 | score salvati=170/388 | 6.7s
[5/113] imb_00978/M1: OK | score=0.108954 | score salvati=171/388 | 4.6s
[6/113] imb_00887/M0: OK | score=0.487307 | score salvati=172/388 | 4.5s
[6/113] imb_00887/M1: OK | score=0.000000 | score salvati=173/388 | 4.4s
[7/113] imb_00103/M0: OK | score=0.5264

### 6. Gate finale degli output RAGAS

Sono attesi 387 score validi: 194 per M0 e 193 per M1.  
Il caso `imb_00791/M1` è registrato separatamente come failure del valutatore causato da una generazione degenerata di M1; non viene imputato alcuno score. La contabilità complessiva deve quindi risultare pari a 387 score validi più un failure documentato, per un totale di 388 esiti

La differenza è definita come M1−M0. L’intervallo di confidenza è ottenuto con bootstrap paired; il test è Wilcoxon signed-rank e l’effect size è la correlazione rank-biserial paired.

Sono attesi 387 score validi: 194 per M0 e 193 per M1.  
Il caso `imb_00791/M1` è registrato separatamente come failure del valutatore causato da una generazione degenerata di M1; non viene imputato alcuno score. La contabilità complessiva deve quindi risultare pari a 387 score validi più un failure documentato, per un totale di 388 esiti.

In [18]:
# Validazione finale degli output RAGAS M0/M1

import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

from collections import Counter

completed = load_completed_scores()
failures = load_recorded_failures()

score_counts = Counter(
    row["condition"] for row in completed.values()
)
failure_counts = Counter(
    row["condition"] for row in failures.values()
)

paired_score_ids = {
    sid
    for sid, condition in completed
    if condition == "M0"
} & {
    sid
    for sid, condition in completed
    if condition == "M1"
}

require(len(completed) == 387, "Numero score validi inatteso")
require(score_counts == {"M0": 194, "M1": 193}, "Conteggi M0/M1 inattesi")
require(len(failures) == 1, "Numero failure inatteso")
require(
    set(failures) == {("imb_00791", "M1")},
    "Failure diverso da quello documentato",
)
require(len(paired_score_ids) == 193, "Numero confronti paired inatteso")
require(len(completed) + len(failures) == 388, "Contabilità incompleta")

print("VALIDAZIONE: PASS")
print("Score validi:", len(completed))
print("  M0:", score_counts["M0"])
print("  M1:", score_counts["M1"])
print("Failure documentati:", len(failures))
print("Confronti paired disponibili:", len(paired_score_ids))
print("SHA256 score:", sha256_file(SCORES_PATH))
print("SHA256 failure:", sha256_file(FAILURES_PATH))

VALIDAZIONE: PASS
Score validi: 387
  M0: 194
  M1: 193
Failure documentati: 1
Confronti paired disponibili: 193
SHA256 score: 06cd711dce97901d8b32615ce305b7429c6c4dface9204d879ab851c5262f1d6
SHA256 failure: f8af57fe741061eda3f6649cc05696bca0a07e9e6d9d18c1b2a81cf95a86543b


## 7. Analisi statistica paired M1−M0

L’analisi utilizza esclusivamente i 193 casi con score disponibili per entrambe le condizioni. La differenza è definita come M1−M0. Sono riportati la differenza media e mediana, l’intervallo di confidenza bootstrap paired al 95%, il test di Wilcoxon signed-rank, la correlazione rank-biserial paired e il conteggio win/tie/loss. Il caso mancante non viene imputato.

In [20]:
# Analisi statistica paired M1 − M0

import csv
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon, rankdata

ANALYSIS_DIR = SCORES_PATH.parent
PAIRED_RESULTS_PATH = ANALYSIS_DIR / "response_relevancy_paired_differences.jsonl"
SUMMARY_CSV_PATH = ANALYSIS_DIR / "response_relevancy_statistical_summary.csv"
SUMMARY_JSON_PATH = ANALYSIS_DIR / "response_relevancy_statistical_summary.json"

BOOTSTRAP_REPETITIONS = 20000
BOOTSTRAP_SEED = 20260831
TIE_EPSILON = 1e-12

completed = load_completed_scores()

paired_rows = []

for pair in eligible_pairs:
    sid = pair["sample_id"]
    key_m0 = (sid, "M0")
    key_m1 = (sid, "M1")

    if key_m0 not in completed or key_m1 not in completed:
        continue

    m0 = float(completed[key_m0]["response_relevancy"])
    m1 = float(completed[key_m1]["response_relevancy"])

    paired_rows.append({
        "sample_id": sid,
        "benchmark_order": pair["benchmark_order"],
        "dataset": pair["dataset"],
        "M0_response_relevancy": m0,
        "M1_response_relevancy": m1,
        "delta_M1_minus_M0": m1 - m0,
    })

require(len(paired_rows) == 193, "Numero paired inatteso")


def bootstrap_mean_ci(values, repetitions, seed):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)

    indices = rng.integers(
        0,
        len(values),
        size=(repetitions, len(values)),
    )

    bootstrap_means = values[indices].mean(axis=1)

    return (
        float(np.quantile(bootstrap_means, 0.025)),
        float(np.quantile(bootstrap_means, 0.975)),
    )


def matched_rank_biserial(delta):
    delta = np.asarray(delta, dtype=float)
    nonzero = delta[np.abs(delta) > TIE_EPSILON]

    if len(nonzero) == 0:
        return 0.0

    ranks = rankdata(np.abs(nonzero), method="average")
    positive_sum = float(ranks[nonzero > 0].sum())
    negative_sum = float(ranks[nonzero < 0].sum())

    return (
        (positive_sum - negative_sum)
        / (positive_sum + negative_sum)
    )


def summarize_group(name, records, seed):
    m0 = np.asarray(
        [row["M0_response_relevancy"] for row in records],
        dtype=float,
    )
    m1 = np.asarray(
        [row["M1_response_relevancy"] for row in records],
        dtype=float,
    )
    delta = m1 - m0

    wins = int(np.sum(delta > TIE_EPSILON))
    losses = int(np.sum(delta < -TIE_EPSILON))
    ties = int(len(delta) - wins - losses)

    ci_low, ci_high = bootstrap_mean_ci(
        delta,
        BOOTSTRAP_REPETITIONS,
        seed,
    )

    if np.all(np.abs(delta) <= TIE_EPSILON):
        wilcoxon_statistic = 0.0
        wilcoxon_p = 1.0
    else:
        test = wilcoxon(
            delta,
            zero_method="wilcox",
            correction=False,
            alternative="two-sided",
            method="approx",
        )
        wilcoxon_statistic = float(test.statistic)
        wilcoxon_p = float(test.pvalue)

    return {
        "group": name,
        "n_paired": int(len(records)),
        "M0_mean": float(m0.mean()),
        "M1_mean": float(m1.mean()),
        "mean_delta_M1_minus_M0": float(delta.mean()),
        "median_delta_M1_minus_M0": float(np.median(delta)),
        "bootstrap_95CI_low": ci_low,
        "bootstrap_95CI_high": ci_high,
        "wilcoxon_statistic": wilcoxon_statistic,
        "wilcoxon_p_two_sided": wilcoxon_p,
        "matched_rank_biserial": matched_rank_biserial(delta),
        "M1_wins": wins,
        "ties": ties,
        "M1_losses": losses,
        "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
        "bootstrap_seed": seed,
    }


groups = [
    ("Overall", paired_rows),
    (
        "IMB_Curated_Test_Set_1000",
        [
            row for row in paired_rows
            if row["dataset"] == "IMB_Curated_Test_Set_1000"
        ],
    ),
    (
        "MMMED_Italian",
        [
            row for row in paired_rows
            if row["dataset"] == "MMMED_Italian"
        ],
    ),
]

summaries = [
    summarize_group(name, records, BOOTSTRAP_SEED + index)
    for index, (name, records) in enumerate(groups)
]

require(summaries[0]["n_paired"] == 193, "Overall n inatteso")
require(summaries[1]["n_paired"] == 98, "IMB n inatteso")
require(summaries[2]["n_paired"] == 95, "MMMED n inatteso")

with PAIRED_RESULTS_PATH.open("w", encoding="utf-8") as f:
    for row in paired_rows:
        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
                sort_keys=True,
            )
            + "\n"
        )

summary_df = pd.DataFrame(summaries)
summary_df.to_csv(SUMMARY_CSV_PATH, index=False)

summary_payload = {
    "analysis": "M1 minus M0 Response Relevancy",
    "score_file_sha256": sha256_file(SCORES_PATH),
    "failure_file_sha256": sha256_file(FAILURES_PATH),
    "n_scores": 387,
    "n_failures": 1,
    "n_paired": 193,
    "missing_pair": {
        "sample_id": "imb_00791",
        "condition": "M1",
        "score_imputed": False,
    },
    "groups": summaries,
}

SUMMARY_JSON_PATH.write_text(
    json.dumps(
        summary_payload,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print("ANALISI STATISTICA: PASS")
display(
    summary_df[
        [
            "group",
            "n_paired",
            "M0_mean",
            "M1_mean",
            "mean_delta_M1_minus_M0",
            "bootstrap_95CI_low",
            "bootstrap_95CI_high",
            "wilcoxon_p_two_sided",
            "matched_rank_biserial",
            "M1_wins",
            "ties",
            "M1_losses",
        ]
    ].round(6)
)

print("\nOutput:")
print(PAIRED_RESULTS_PATH)
print(SUMMARY_CSV_PATH)
print(SUMMARY_JSON_PATH)

ANALISI STATISTICA: PASS


,group,n_paired,M0_mean,M1_mean,mean_delta_M1_minus_M0,bootstrap_95CI_low,bootstrap_95CI_high,wilcoxon_p_two_sided,matched_rank_biserial,M1_wins,ties,M1_losses
0,Overall,193,0.440699,0.230307,-0.210392,-0.255411,-0.164359,0.000000,-0.625537,53,13,127
1,IMB_Curated_Test_Set_1000,98,0.465266,0.148431,-0.316835,-0.372477,-0.258376,0.000000,-0.822172,16,4,78
2,MMMED_Italian,95,0.415355,0.314769,-0.100587,-0.165196,-0.037073,0.018185,-0.293237,37,9,49



Output:
/home/eugrug/Project_CRAG_MedGemma/benchmark_m0_m1/ragas_m0_m1_v07_03/response_relevancy_paired_differences.jsonl
/home/eugrug/Project_CRAG_MedGemma/benchmark_m0_m1/ragas_m0_m1_v07_03/response_relevancy_statistical_summary.csv
/home/eugrug/Project_CRAG_MedGemma/benchmark_m0_m1/ragas_m0_m1_v07_03/response_relevancy_statistical_summary.json


### 8. Verifica dei p-value senza arrotondamento

I p-value del test di Wilcoxon vengono visualizzati in notazione scientifica per evitare che valori molto piccoli siano erroneamente riportati come zero a causa dell’arrotondamento tabellare. Questa verifica non modifica l’analisi e non effettua nuove valutazioni: legge esclusivamente i risultati statistici già calcolati e consente di registrare nel paper i valori esatti.

In [21]:
print("P-value esatti in notazione scientifica:")
for row in summaries:
    print(
        f"{row['group']}: "
        f"p={row['wilcoxon_p_two_sided']:.12e}"
    )

P-value esatti in notazione scientifica:
Overall: p=3.384429524391e-13
IMB_Curated_Test_Set_1000: p=4.467275158132e-12
MMMED_Italian: p=1.818452170317e-02


### Esito dell’ipotesi H1a

L’ipotesi H1a non è supportata. Rispetto al modello base M0, il modello fine-tuned M1 presenta una riduzione significativa della Response Relevancy complessiva (differenza media M1−M0 = −0,210; IC bootstrap 95% [−0,255; −0,164]; Wilcoxon signed-rank, p = 3,38×10⁻¹³; correlazione rank-biserial paired = −0,626). Il peggioramento è più marcato nella coorte IMB e rimane osservabile nella coorte MMMED. M1 registra 53 vittorie, 13 pareggi e 127 sconfitte rispetto a M0. Un’ulteriore generazione degenerata di M1 non ha ricevuto uno score imputato ed è stata esclusa dal confronto paired. Questi risultati riguardano esclusivamente l’adapter FT-10K nelle condizioni sperimentali congelate e non anticipano l’esito del confronto R0−R1 nella pipeline EMCRAG.